In [ ]:
# Bỏ dấu # nếu môi trường chưa có thư viện.
# %pip install -q pandas numpy scikit-learn tqdm

## 1. Cấu hình

In [2]:
from __future__ import annotations

import hashlib
import json
import math
import random
import re
import unicodedata
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable, Iterable, Sequence

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from tqdm.auto import tqdm


# ============================================================
# ĐƯỜNG DẪN
# ============================================================

DATA_DIR = Path("D:\\KLTN\\code\\data\\processed")

DOMAIN_FILES = {
    "education": DATA_DIR / "education_clean.json",
    "finance": DATA_DIR / "finance_clean.json",
    "legal": DATA_DIR / "legal_clean.json",
    "medical": DATA_DIR / "medical_clean.json",
    "news": DATA_DIR / "news_clean.json",
    "science": DATA_DIR / "science_clean.json",
}

OUTPUT_DIR = Path("./six_domain_binary_output")
AUDIT_DIR = OUTPUT_DIR / "01_accept_reject"
FROZEN_DIR = OUTPUT_DIR / "02_frozen"
SPLIT_DIR = OUTPUT_DIR / "03_splits"
PILOT_DIR = OUTPUT_DIR / "04_pilot"

for directory in [AUDIT_DIR, FROZEN_DIR, SPLIT_DIR, PILOT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


# ============================================================
# QUY TẮC ACCEPT / REJECT
# ============================================================

# Domain trong file nguồn được xem là domain chuẩn.
# Nếu field domain bên trong record khác tên file, notebook sửa về tên file,
# không reject chỉ vì mismatch.
VALID_DOMAINS = set(DOMAIN_FILES)

# Ngưỡng đủ dùng tối thiểu. Có thể giảm nếu corpus có nhiều văn bản ngắn hợp lệ.
MIN_WORDS_BY_DOMAIN = {
    "education": 30,
    "finance": 50,
    "legal": 25,
    "medical": 50,
    "news": 60,
    "science": 60,
}

MIN_CHARS = 80
MIN_ALPHA_RATIO = 0.45
MAX_REPEATED_PARAGRAPH_RATIO = 0.35

# Chỉ reject boilerplate khi xuất hiện nhiều dấu hiệu cùng lúc.
BOILERPLATE_PATTERNS = [
    r"\btin liên quan\b",
    r"\bxem thêm\b",
    r"\bchia sẻ bài viết\b",
    r"\btheo dõi chúng tôi\b",
    r"\bđăng ký nhận tin\b",
    r"\bquảng cáo\b",
    r"\bcookie\b",
    r"\bmenu\b",
    r"\bfooter\b",
]
MAX_BOILERPLATE_HITS = 2

# Exact duplicate: giữ bản dài nhất; nếu bằng nhau, giữ ID nhỏ hơn.
REMOVE_EXACT_DUPLICATES = True


# ============================================================
# FREEZE + SPLIT
# ============================================================

CORPUS_VERSION = "contexts_v1"
TRAIN_RATIO = 0.80
DEV_RATIO = 0.10
TEST_RATIO = 0.10
RANDOM_SEED = 42


# ============================================================
# PILOT
# ============================================================

PILOT_CONTEXTS_PER_DOMAIN = {
    "train": 30,
    "dev": 10,
    "test": 10,
}

MULTI_DOC_REQUESTS_PER_TYPE = 3

assert math.isclose(TRAIN_RATIO + DEV_RATIO + TEST_RATIO, 1.0)

print("Data   :", DATA_DIR.resolve())
print("Output :", OUTPUT_DIR.resolve())

Data   : D:\KLTN\code\data\processed
Output : D:\KLTN\code\notebooks\six_domain_binary_output


## 2. Hàm đọc, ghi và normalize

In [3]:
ZERO_WIDTH_RE = re.compile(r"[\u200B-\u200D\uFEFF]")
WORD_RE = re.compile(r"\b\w+\b", flags=re.UNICODE)
MULTISPACE_RE = re.compile(r"[ \t]+")
EXCESS_NEWLINES_RE = re.compile(r"\n{3,}")


def read_json_or_jsonl(path: Path) -> list[Any]:
    if not path.exists():
        raise FileNotFoundError(f"Không tìm thấy: {path.resolve()}")

    text = path.read_text(encoding="utf-8-sig")

    if not text.strip():
        return []

    try:
        data = json.loads(text)
        if isinstance(data, list):
            return data
        if isinstance(data, dict):
            return [data]
        raise ValueError(f"JSON root không hợp lệ: {path.name}")
    except json.JSONDecodeError:
        records = []

        for line_number, line in enumerate(text.splitlines(), start=1):
            line = line.strip()
            if not line:
                continue

            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"JSONL lỗi tại {path.name}, dòng {line_number}: {exc}"
                ) from exc

        return records


def write_jsonl(records: Iterable[dict[str, Any]], path: Path) -> int:
    path.parent.mkdir(parents=True, exist_ok=True)
    count = 0

    with path.open("w", encoding="utf-8") as file:
        for record in records:
            file.write(json.dumps(record, ensure_ascii=False) + "\n")
            count += 1

    return count


def normalize_content(text: str) -> str:
    text = unicodedata.normalize("NFC", text)
    text = ZERO_WIDTH_RE.sub("", text)
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    lines = [
        MULTISPACE_RE.sub(" ", line).strip()
        for line in text.split("\n")
    ]

    text = "\n".join(lines)
    text = EXCESS_NEWLINES_RE.sub("\n\n", text)
    return text.strip()


def normalize_for_hash(text: str) -> str:
    text = unicodedata.normalize("NFKC", text).lower()
    text = ZERO_WIDTH_RE.sub("", text)
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    return re.sub(r"\s+", " ", text).strip()


def content_hash(text: str) -> str:
    normalized = normalize_for_hash(text)
    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()


def stable_hash(value: str, seed: int = 0) -> str:
    return hashlib.sha256(
        f"{seed}:{value}".encode("utf-8")
    ).hexdigest()


def count_words(text: str) -> int:
    return len(WORD_RE.findall(text))


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)

    return digest.hexdigest()


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()

## 3. Đọc sáu file

In [4]:
loaded_records: list[dict[str, Any]] = []

for expected_domain, path in DOMAIN_FILES.items():
    raw_records = read_json_or_jsonl(path)

    for source_index, raw_record in enumerate(raw_records):
        loaded_records.append({
            "raw": raw_record,
            "expected_domain": expected_domain,
            "source_file": path.name,
            "source_index": source_index,
        })

    print(
        f"{expected_domain:10s}: "
        f"{len(raw_records):,} records"
    )

print("\nTổng:", f"{len(loaded_records):,}")

education : 1,636 records
finance   : 5,467 records
legal     : 2,085 records
medical   : 4,348 records
news      : 5,379 records
science   : 1,517 records

Tổng: 20,432


## 4. Auto Accept/Reject

Reject khi gặp một trong các lỗi mạnh:

- Record không phải object.
- Thiếu `id`, `content` hoặc `domain`.
- ID hoặc content rỗng.
- Content không phải string.
- Quá ngắn theo domain.
- Tỷ lệ ký tự chữ quá thấp.
- Paragraph lặp quá nhiều.
- Còn quá nhiều dấu hiệu menu/footer/quảng cáo.
- ID trùng.
- Exact duplicate không phải canonical.

Các document dài bất thường không bị reject tự động.

In [5]:
def alphabetic_ratio(text: str) -> float:
    visible = [char for char in text if not char.isspace()]

    if not visible:
        return 0.0

    return sum(char.isalpha() for char in visible) / len(visible)


def repeated_paragraph_ratio(text: str) -> float:
    paragraphs = [
        normalize_for_hash(paragraph)
        for paragraph in re.split(r"\n\s*\n|\n", text)
        if normalize_for_hash(paragraph)
    ]

    if len(paragraphs) <= 1:
        return 0.0

    counts = Counter(paragraphs)
    repeated = sum(
        frequency - 1
        for frequency in counts.values()
        if frequency > 1
    )

    return repeated / len(paragraphs)


def count_boilerplate_hits(text: str) -> int:
    normalized = text.lower()

    return sum(
        1
        for pattern in BOILERPLATE_PATTERNS
        if re.search(pattern, normalized, flags=re.IGNORECASE)
    )


def initial_binary_audit(
    loaded: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    audited = []

    for item in tqdm(loaded, desc="Auto audit"):
        raw = item["raw"]
        domain = item["expected_domain"]

        reasons: list[str] = []

        if not isinstance(raw, dict):
            audited.append({
                "id": f"invalid_{domain}_{item['source_index']}",
                "content": "",
                "domain": domain,
                "source_file": item["source_file"],
                "source_index": item["source_index"],
                "status": "rejected",
                "reject_reasons": ["record_not_object"],
                "word_count": 0,
                "char_count": 0,
                "alpha_ratio": 0.0,
                "repeated_paragraph_ratio": 0.0,
                "boilerplate_hits": 0,
                "content_hash": "",
            })
            continue

        missing = [
            field
            for field in ("id", "content", "domain")
            if field not in raw
        ]

        if missing:
            reasons.append("missing_fields:" + ",".join(missing))

        doc_id = str(raw.get("id", "")).strip()
        raw_content = raw.get("content", "")

        if not doc_id:
            reasons.append("empty_id")

        if not isinstance(raw_content, str):
            reasons.append("content_not_string")
            text = (
                str(raw_content)
                if raw_content is not None
                else ""
            )
        else:
            text = raw_content

        text = normalize_content(text)

        words = count_words(text)
        chars = len(text)
        alpha = alphabetic_ratio(text)
        repeat_ratio = repeated_paragraph_ratio(text)
        boilerplate_hits = count_boilerplate_hits(text)

        if not text:
            reasons.append("empty_content")

        if chars < MIN_CHARS:
            reasons.append("content_too_short_chars")

        if words < MIN_WORDS_BY_DOMAIN[domain]:
            reasons.append("content_too_short_for_domain")

        if alpha < MIN_ALPHA_RATIO:
            reasons.append("low_alphabetic_ratio")

        if repeat_ratio > MAX_REPEATED_PARAGRAPH_RATIO:
            reasons.append("too_many_repeated_paragraphs")

        if boilerplate_hits > MAX_BOILERPLATE_HITS:
            reasons.append("too_many_boilerplate_signals")

        audited.append({
            "id": doc_id,
            "content": text,
            "domain": domain,
            "source_file": item["source_file"],
            "source_index": item["source_index"],
            "status": "rejected" if reasons else "accepted",
            "reject_reasons": sorted(set(reasons)),
            "word_count": words,
            "char_count": chars,
            "alpha_ratio": round(alpha, 4),
            "repeated_paragraph_ratio": round(repeat_ratio, 4),
            "boilerplate_hits": boilerplate_hits,
            "content_hash": content_hash(text) if text else "",
        })

    return audited


audited_records = initial_binary_audit(loaded_records)

Counter(record["status"] for record in audited_records)

Auto audit: 100%|██████████| 20432/20432 [03:43<00:00, 91.55it/s] 


Counter({'accepted': 20371, 'rejected': 61})

## 5. Xử lý ID trùng và exact duplicate

In [6]:
def reject_duplicate_ids(
    records: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    groups: dict[str, list[int]] = defaultdict(list)

    for index, record in enumerate(records):
        if record["id"]:
            groups[record["id"]].append(index)

    updated = [dict(record) for record in records]

    for doc_id, indices in groups.items():
        if len(indices) <= 1:
            continue

        # Giữ bản accepted dài nhất; nếu bằng nhau giữ source index nhỏ hơn.
        canonical_index = sorted(
            indices,
            key=lambda index: (
                updated[index]["status"] != "accepted",
                -updated[index]["word_count"],
                updated[index]["source_file"],
                updated[index]["source_index"],
            ),
        )[0]

        for index in indices:
            if index == canonical_index:
                continue

            updated[index]["status"] = "rejected"
            updated[index]["reject_reasons"] = sorted(
                set(updated[index]["reject_reasons"])
                | {"duplicate_id"}
            )

    return updated


def reject_exact_duplicates(
    records: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    if not REMOVE_EXACT_DUPLICATES:
        return records

    groups: dict[str, list[int]] = defaultdict(list)

    for index, record in enumerate(records):
        if (
            record["status"] == "accepted"
            and record["content_hash"]
        ):
            groups[record["content_hash"]].append(index)

    updated = [dict(record) for record in records]

    for hash_value, indices in groups.items():
        if len(indices) <= 1:
            continue

        canonical_index = sorted(
            indices,
            key=lambda index: (
                -updated[index]["word_count"],
                updated[index]["id"],
            ),
        )[0]

        for index in indices:
            if index == canonical_index:
                continue

            updated[index]["status"] = "rejected"
            updated[index]["reject_reasons"] = sorted(
                set(updated[index]["reject_reasons"])
                | {"exact_duplicate_noncanonical"}
            )
            updated[index]["canonical_id"] = updated[
                canonical_index
            ]["id"]

    return updated


audited_records = reject_duplicate_ids(audited_records)
audited_records = reject_exact_duplicates(audited_records)

accepted_records = [
    record
    for record in audited_records
    if record["status"] == "accepted"
]

rejected_records = [
    record
    for record in audited_records
    if record["status"] == "rejected"
]

print("Accepted:", f"{len(accepted_records):,}")
print("Rejected:", f"{len(rejected_records):,}")

Accepted: 20,371
Rejected: 61


## 6. Xuất accepted/rejected và báo cáo

In [7]:
def compact_document(record: dict[str, Any]) -> dict[str, Any]:
    return {
        "id": record["id"],
        "content": record["content"],
        "domain": record["domain"],
    }


write_jsonl(
    [compact_document(record) for record in accepted_records],
    AUDIT_DIR / "accepted.jsonl",
)

write_jsonl(
    [
        {
            **compact_document(record),
            "reject_reasons": record["reject_reasons"],
            "canonical_id": record.get("canonical_id"),
        }
        for record in rejected_records
    ],
    AUDIT_DIR / "rejected.jsonl",
)


audit_rows = []
for record in audited_records:
    audit_rows.append({
        "id": record["id"],
        "domain": record["domain"],
        "source_file": record["source_file"],
        "status": record["status"],
        "reject_reasons": "|".join(record["reject_reasons"]),
        "word_count": record["word_count"],
        "char_count": record["char_count"],
        "alpha_ratio": record["alpha_ratio"],
        "repeated_paragraph_ratio":
            record["repeated_paragraph_ratio"],
        "boilerplate_hits": record["boilerplate_hits"],
        "content_preview": record["content"][:1000],
    })

audit_df = pd.DataFrame(audit_rows)

audit_df.to_csv(
    AUDIT_DIR / "full_accept_reject_audit.csv",
    index=False,
    encoding="utf-8-sig",
)


domain_summary = (
    audit_df.groupby(["domain", "status"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

domain_summary.to_csv(
    AUDIT_DIR / "domain_accept_reject_summary.csv",
    index=False,
    encoding="utf-8-sig",
)


reason_counter = Counter()
for record in rejected_records:
    for reason in record["reject_reasons"]:
        reason_counter[(record["domain"], reason)] += 1

reject_reason_df = pd.DataFrame([
    {
        "domain": domain,
        "reject_reason": reason,
        "count": count,
    }
    for (domain, reason), count in sorted(reason_counter.items())
])

reject_reason_df.to_csv(
    AUDIT_DIR / "reject_reason_statistics.csv",
    index=False,
    encoding="utf-8-sig",
)


# Spot-check nhỏ, không ảnh hưởng pipeline.
rng = random.Random(RANDOM_SEED)
spot_check_rows = []

for domain in sorted(VALID_DOMAINS):
    for status, records in [
        ("accepted", accepted_records),
        ("rejected", rejected_records),
    ]:
        candidates = [
            record
            for record in records
            if record["domain"] == domain
        ]

        sample_size = min(50, len(candidates))
        sampled = (
            rng.sample(candidates, sample_size)
            if sample_size
            else []
        )

        for record in sampled:
            spot_check_rows.append({
                "domain": domain,
                "status": status,
                "id": record["id"],
                "reject_reasons": "|".join(
                    record["reject_reasons"]
                ),
                "word_count": record["word_count"],
                "content_preview": record["content"][:1500],
            })

pd.DataFrame(spot_check_rows).to_csv(
    AUDIT_DIR / "optional_spot_check.csv",
    index=False,
    encoding="utf-8-sig",
)

display(domain_summary)
display(reject_reason_df.head(30))

status,domain,accepted,rejected
0,education,1595,41
1,finance,5466,1
2,legal,2079,6
3,medical,4345,3
4,news,5372,7
5,science,1514,3


,domain,reject_reason,count
0,education,content_too_short_for_domain,1
1,education,low_alphabetic_ratio,38
2,education,too_many_boilerplate_signals,3
3,finance,content_too_short_for_domain,1
4,legal,low_alphabetic_ratio,6
5,medical,content_too_short_for_domain,3
6,news,content_too_short_for_domain,7
7,science,too_many_repeated_paragraphs,3


# Pha B — Freeze corpus tự động

Không có manual decision. Tất cả record `accepted` được đóng băng trực tiếp.

In [8]:
frozen_records = []

for record in accepted_records:
    frozen_records.append({
        "id": record["id"],
        "content": record["content"],
        "domain": record["domain"],
        "content_hash": record["content_hash"],
        "context_group_id": (
            "ctxgrp_" + stable_hash(
                record["content_hash"],
                RANDOM_SEED,
            )[:16]
        ),
    })

frozen_records.sort(
    key=lambda record: (
        record["domain"],
        record["id"],
    )
)

frozen_path = FROZEN_DIR / f"{CORPUS_VERSION}.jsonl"
write_jsonl(frozen_records, frozen_path)

manifest = {
    "corpus_version": CORPUS_VERSION,
    "created_at_utc": utc_now_iso(),
    "total_contexts": len(frozen_records),
    "domain_counts": dict(
        sorted(
            Counter(
                record["domain"]
                for record in frozen_records
            ).items()
        )
    ),
    "file": frozen_path.name,
    "file_sha256": file_sha256(frozen_path),
    "policy": "automatic_binary_accept_reject",
}

with (FROZEN_DIR / "manifest.json").open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(manifest, file, ensure_ascii=False, indent=2)

print(json.dumps(manifest, ensure_ascii=False, indent=2))

{
  "corpus_version": "contexts_v1",
  "created_at_utc": "2026-08-03T18:26:28.458393+00:00",
  "total_contexts": 20371,
  "domain_counts": {
    "education": 1595,
    "finance": 5466,
    "legal": 2079,
    "medical": 4345,
    "news": 5372,
    "science": 1514
  },
  "file": "contexts_v1.jsonl",
  "file_sha256": "005fec6029bda580116cb390bfeec38bbe1f686a9f45a2cf2d6bb0e23d95e8ee",
  "policy": "automatic_binary_accept_reject"
}


# Pha C — Context-level split

In [9]:
def allocate_counts(n: int) -> dict[str, int]:
    raw = {
        "train": n * TRAIN_RATIO,
        "dev": n * DEV_RATIO,
        "test": n * TEST_RATIO,
    }

    counts = {
        split: int(math.floor(value))
        for split, value in raw.items()
    }

    remaining = n - sum(counts.values())

    fractional_order = sorted(
        raw,
        key=lambda split: raw[split] - counts[split],
        reverse=True,
    )

    for split in fractional_order[:remaining]:
        counts[split] += 1

    if n >= 10:
        for target in ("dev", "test"):
            if counts[target] == 0 and counts["train"] > 1:
                counts[target] += 1
                counts["train"] -= 1

    return counts


def split_contexts(
    records: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    output = []

    for domain in sorted(VALID_DOMAINS):
        domain_records = [
            record
            for record in records
            if record["domain"] == domain
        ]

        domain_records = sorted(
            domain_records,
            key=lambda record: stable_hash(
                f"{domain}:{record['context_group_id']}",
                RANDOM_SEED,
            ),
        )

        counts = allocate_counts(len(domain_records))
        train_end = counts["train"]
        dev_end = train_end + counts["dev"]

        for index, record in enumerate(domain_records):
            if index < train_end:
                split = "train"
            elif index < dev_end:
                split = "dev"
            else:
                split = "test"

            output.append({
                **record,
                "split": split,
            })

    return output


split_records = split_contexts(frozen_records)

records_by_split = {
    split: [
        record
        for record in split_records
        if record["split"] == split
    ]
    for split in ("train", "dev", "test")
}

for split, records in records_by_split.items():
    write_jsonl(
        records,
        SPLIT_DIR / f"contexts_{split}.jsonl",
    )


def assert_no_leakage(
    records_by_split: dict[str, list[dict[str, Any]]],
) -> None:
    split_names = list(records_by_split)

    for field in ("id", "content_hash", "context_group_id"):
        values = {
            split: {
                record[field]
                for record in records
            }
            for split, records in records_by_split.items()
        }

        for left_index, left in enumerate(split_names):
            for right in split_names[left_index + 1:]:
                overlap = values[left] & values[right]

                if overlap:
                    raise AssertionError(
                        f"Leakage {field}: {left} ↔ {right}"
                    )


assert_no_leakage(records_by_split)

split_rows = []
for domain in sorted(VALID_DOMAINS):
    for split in ("train", "dev", "test"):
        split_rows.append({
            "domain": domain,
            "split": split,
            "count": sum(
                1
                for record in split_records
                if (
                    record["domain"] == domain
                    and record["split"] == split
                )
            ),
        })

split_df = pd.DataFrame(split_rows)
split_df.to_csv(
    SPLIT_DIR / "split_statistics.csv",
    index=False,
    encoding="utf-8-sig",
)

display(
    split_df.pivot(
        index="domain",
        columns="split",
        values="count",
    ).fillna(0).astype(int)
)

split,dev,test,train
domain,,,
education,160,159,1276
finance,547,546,4373
legal,208,208,1663
medical,435,434,3476
news,537,537,4298
science,152,151,1211


# Pha D — Pilot data generation

In [10]:
SUMMARY_GUIDANCE = {
    "education": (
        "Tạo yêu cầu tóm tắt tập trung vào đối tượng, điều kiện, "
        "mốc thời gian, yêu cầu và kết quả giáo dục."
    ),
    "finance": (
        "Tạo yêu cầu tóm tắt tập trung vào diễn biến, số liệu, "
        "nguyên nhân và tác động tài chính."
    ),
    "legal": (
        "Tạo yêu cầu tóm tắt tập trung vào quy định, đối tượng áp dụng, "
        "quyền, nghĩa vụ, điều kiện và ngoại lệ."
    ),
    "medical": (
        "Tạo yêu cầu tóm tắt tập trung vào thông tin y khoa, đối tượng, "
        "biểu hiện, xử lý và khuyến cáo."
    ),
    "news": (
        "Tạo yêu cầu tóm tắt tập trung vào sự kiện, các bên liên quan, "
        "diễn biến, số liệu và kết quả."
    ),
    "science": (
        "Tạo yêu cầu tóm tắt tập trung vào mục tiêu, phương pháp, "
        "kết quả và kết luận."
    ),
}


def select_pilot_contexts() -> list[dict[str, Any]]:
    selected = []

    for split, limit in PILOT_CONTEXTS_PER_DOMAIN.items():
        for domain in sorted(VALID_DOMAINS):
            candidates = [
                record
                for record in records_by_split[split]
                if record["domain"] == domain
            ]

            candidates = sorted(
                candidates,
                key=lambda record: stable_hash(
                    f"pilot:{split}:{domain}:{record['id']}",
                    RANDOM_SEED,
                ),
            )

            selected.extend(candidates[:min(limit, len(candidates))])

    return selected


pilot_contexts = select_pilot_contexts()

write_jsonl(
    pilot_contexts,
    PILOT_DIR / "pilot_contexts.jsonl",
)

print("Pilot contexts:", len(pilot_contexts))

Pilot contexts: 300


In [11]:
def format_contexts(
    contexts: Sequence[dict[str, Any]],
) -> str:
    blocks = []

    for index, context in enumerate(contexts, start=1):
        blocks.append(
            f"[CONTEXT {index}]\n"
            f"context_id: {context['id']}\n"
            f"domain: {context['domain']}\n"
            f"{context['content']}"
        )

    return "\n\n".join(blocks)


def qa_prompt(
    contexts: Sequence[dict[str, Any]],
    question_type: str,
) -> str:
    requirement = {
        "vanilla": (
            "Tạo một câu hỏi về một sự thật trực tiếp có thể "
            "kiểm chứng từ context."
        ),
        "multi_hop": (
            "Tạo câu hỏi cần kết nối ít nhất hai mảnh bằng chứng. "
            "Nếu bundle không phù hợp, trả valid=false."
        ),
        "comparison": (
            "Tạo câu hỏi so sánh hai thực thể, sự kiện, số liệu hoặc "
            "quy định. Nếu không có cơ sở hợp lý, trả valid=false."
        ),
        "set_operation": (
            "Tạo câu hỏi liệt kê các phần tử thỏa mãn nhiều điều kiện. "
            "Nếu không thể tạo tự nhiên, trả valid=false."
        ),
    }[question_type]

    return f"""Bạn đang tạo dữ liệu QA tiếng Việt.

Loại câu hỏi: {question_type}
Yêu cầu: {requirement}

Quy tắc:
- Chỉ dùng thông tin trong context.
- Không dùng kiến thức ngoài.
- Câu trả lời phải ngắn gọn nhưng đầy đủ.
- Evidence phải là trích đoạn có thật.
- Chỉ trả một JSON object hợp lệ.

Schema:
{{
  "valid": true,
  "question": "...",
  "answer": "...",
  "evidence": [
    {{"context_id": "...", "quote": "..."}}
  ],
  "reason_if_invalid": ""
}}

Context:
{format_contexts(contexts)}
"""


def summary_prompt(context: dict[str, Any]) -> str:
    return f"""Bạn đang tạo dữ liệu summarization tiếng Việt.

Định hướng:
{SUMMARY_GUIDANCE[context['domain']]}

Quy tắc:
- Tạo một instruction tóm tắt tự nhiên.
- Summary chỉ dùng thông tin trong context.
- Không thêm kiến thức ngoài.
- Giữ số liệu, điều kiện và quan hệ quan trọng.
- Chỉ trả một JSON object hợp lệ.

Schema:
{{
  "valid": true,
  "instruction": "...",
  "summary": "...",
  "evidence_context_ids": ["{context['id']}"],
  "reason_if_invalid": ""
}}

Context:
{format_contexts([context])}
"""

In [12]:
def related_bundles(
    records: list[dict[str, Any]],
    bundle_size: int,
    max_bundles: int,
) -> list[list[dict[str, Any]]]:
    if len(records) < bundle_size:
        return []

    texts = [record["content"] for record in records]

    try:
        matrix = TfidfVectorizer(
            max_features=20_000,
            ngram_range=(1, 2),
            min_df=1,
            max_df=1.0,
            sublinear_tf=True,
        ).fit_transform(texts)
    except ValueError:
        return []

    n_neighbors = min(
        max(bundle_size + 2, 5),
        len(records),
    )

    model = NearestNeighbors(
        n_neighbors=n_neighbors,
        metric="cosine",
        algorithm="brute",
    )
    model.fit(matrix)

    distances, indices = model.kneighbors(matrix)

    candidates = []

    for anchor in range(len(records)):
        related = [
            int(index)
            for index in indices[anchor]
            if int(index) != anchor
        ]

        chosen = [anchor] + related[:bundle_size - 1]

        if len(chosen) == bundle_size:
            candidates.append((
                float(np.mean(distances[anchor][1:bundle_size])),
                chosen,
            ))

    candidates.sort(key=lambda item: item[0])

    bundles = []
    seen = set()

    for _, indices_ in candidates:
        bundle = [records[index] for index in indices_]
        signature = tuple(
            sorted(record["id"] for record in bundle)
        )

        if signature in seen:
            continue

        seen.add(signature)
        bundles.append(bundle)

        if len(bundles) >= max_bundles:
            break

    return bundles


def build_pilot_requests() -> list[dict[str, Any]]:
    requests = []

    for context in pilot_contexts:
        signature = (
            f"{context['split']}:{context['domain']}:{context['id']}"
        )

        requests.append({
            "request_id": "qa_vanilla_" + stable_hash(signature)[:16],
            "split": context["split"],
            "domain": context["domain"],
            "task": "qa",
            "question_type": "vanilla",
            "context_ids": [context["id"]],
            "prompt": qa_prompt([context], "vanilla"),
        })

        requests.append({
            "request_id": "summary_" + stable_hash(signature)[:16],
            "split": context["split"],
            "domain": context["domain"],
            "task": "summarization",
            "question_type": None,
            "context_ids": [context["id"]],
            "prompt": summary_prompt(context),
        })

    bundle_sizes = {
        "multi_hop": 2,
        "comparison": 2,
        "set_operation": 3,
    }

    for split in ("train", "dev", "test"):
        for domain in sorted(VALID_DOMAINS):
            group = [
                context
                for context in pilot_contexts
                if (
                    context["split"] == split
                    and context["domain"] == domain
                )
            ]

            for question_type, bundle_size in bundle_sizes.items():
                bundles = related_bundles(
                    group,
                    bundle_size=bundle_size,
                    max_bundles=MULTI_DOC_REQUESTS_PER_TYPE,
                )

                for bundle in bundles:
                    ids = [context["id"] for context in bundle]
                    signature = (
                        f"{split}:{domain}:{question_type}:"
                        + "|".join(sorted(ids))
                    )

                    requests.append({
                        "request_id": (
                            f"qa_{question_type}_"
                            + stable_hash(signature)[:16]
                        ),
                        "split": split,
                        "domain": domain,
                        "task": "qa",
                        "question_type": question_type,
                        "context_ids": ids,
                        "prompt": qa_prompt(
                            bundle,
                            question_type,
                        ),
                    })

    return requests


pilot_requests = build_pilot_requests()

write_jsonl(
    pilot_requests,
    PILOT_DIR / "pilot_generation_requests.jsonl",
)

request_df = pd.DataFrame([
    {
        "split": request["split"],
        "domain": request["domain"],
        "task": request["task"],
        "question_type": request["question_type"],
    }
    for request in pilot_requests
])

request_statistics = (
    request_df.groupby(
        ["split", "domain", "task", "question_type"],
        dropna=False,
    )
    .size()
    .reset_index(name="count")
)

request_statistics.to_csv(
    PILOT_DIR / "pilot_request_statistics.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Pilot requests:", len(pilot_requests))
display(request_statistics.head(50))

Pilot requests: 762


,split,domain,task,question_type,count
0,dev,education,qa,comparison,3
1,dev,education,qa,multi_hop,3
2,dev,education,qa,set_operation,3
3,dev,education,qa,vanilla,10
4,dev,education,summarization,NaN,10
5,dev,finance,qa,comparison,3
6,dev,finance,qa,multi_hop,3
7,dev,finance,qa,set_operation,3
8,dev,finance,qa,vanilla,10
9,dev,finance,summarization,NaN,10
